# MVSR – Vorbereitung Einheit 1
## OpenCV Basics, Bilddarstellung, Kameramodell und Tiefe

**Bearbeitungszeit:** ca. 30–45 Minuten

Dieses Notebook bereitet die Inhalte der ersten MVSR-Einheit praktisch vor. Sie arbeiten mit **C++ und OpenCV** und untersuchen, wie Bilder, Farbräume, Kameraparameter und Tiefenwerte in Software dargestellt werden.

Für dieses Binder-Notebook wird **OpenCV 4.6** verwendet.

### Lernziele

Nach Bearbeitung dieses Notebooks können Sie ...

- ein Bild mit OpenCV erzeugen und dessen Datenstruktur untersuchen,
- Pixelwerte und Farbkanäle interpretieren,
- zwischen BGR/RGB und HSV konvertieren,
- einfache Farbsegmentierung mit Schwellwerten durchführen,
- die Bedeutung der intrinsischen Kameraparameter erklären,
- einen 3D-Punkt in das Bild projizieren,
- aus Pixelkoordinaten und Tiefe wieder einen 3D-Punkt rekonstruieren.

> **Hinweis:** Dieses Notebook ist als Vorbereitung gedacht. Die vollständige Kamerakalibrierung ist Bestandteil der ersten Hausübung.

### Hinweise zur Verwendung dieses Notebooks

Dieses Notebook besteht aus Text- und Codezellen. Die Textzellen enthalten kurze Erklärungen und Aufgabenstellungen, die C++-Codezellen können direkt ausgeführt und verändert werden.

Eine Codezelle wird über den **Play-Button** oder mit **Shift + Enter** ausgeführt. Die Ausgabe erscheint anschließend direkt unterhalb der jeweiligen Zelle.

Arbeiten Sie das Notebook am besten von oben nach unten durch, da spätere Codezellen teilweise auf zuvor definierten Variablen und Funktionen aufbauen.

Bei Aufgaben mit `TODO` sollen Sie den vorhandenen Code selbstständig ergänzen oder verändern.

C++ wird mit **xeus-cling** inkrementell ausgeführt. Dadurch können bereits deklarierte Variablen bei einer erneuten Ausführung derselben Zelle zu einer Fehlermeldung führen. Wo dies für eine Übungszelle relevant ist, wird im Notebook darauf hingewiesen. Falls nötig, starten Sie den Kernel neu und führen Sie die Zellen erneut von oben nach unten aus.

## 0. Setup

Die Binder-Umgebung ist bereits mit **xeus-cling** und **OpenCV 4.6** vorbereitet.

> **Binder-/Notebook-Hinweis:** Die folgenden `#pragma cling`-Anweisungen sind ein Workaround für den interaktiven C++-Kernel. Sie teilen `xeus-cling` mit, wo die OpenCV-Header und die kompilierten Bibliotheken liegen. In einem normalen lokalen C++-Projekt wird OpenCV stattdessen beim Kompilieren bzw. über das Build-System, z.B. mit CMake, eingebunden und gelinkt. Im eigentlichen C++-Quellcode genügt dann üblicherweise `#include <opencv2/opencv.hpp>`.

In [ ]:
#include <iostream>
#include <iomanip>
#include <vector>
#include <string>
#include <cmath>

// OpenCV-Pfade für Binder / Ubuntu
#pragma cling add_include_path("/usr/include/opencv4")
#pragma cling add_library_path("/usr/lib/x86_64-linux-gnu")

// Benötigte OpenCV-Bibliotheken laden
#pragma cling load("opencv_core")
#pragma cling load("opencv_imgproc")
#pragma cling load("opencv_imgcodecs")
#pragma cling load("opencv_features2d")
#pragma cling load("opencv_calib3d")

#include <opencv2/opencv.hpp>

std::cout << "OpenCV-Version: " << CV_VERSION << std::endl;
std::cout << "C++ Standard: " << __cplusplus << std::endl;

### Hilfsfunktion zur Darstellung

OpenCV verwendet in einem lokalen C++-Programm normalerweise `cv::imshow()` zur Bilddarstellung. Dabei wird ein eigenes Fenster geöffnet, z.B.

```cpp
cv::imshow("Bild", image);
cv::waitKey(0);
```

In Binder läuft das Notebook jedoch im Browser und besitzt keine normale Desktop-GUI.

> **Binder-/Notebook-Hinweis:** Die folgende Hilfsfunktion ist daher ein Workaround für die Browser-Umgebung. Sie kodiert eine `cv::Mat` als PNG und übergibt sie an die Rich-Display-Funktion des Jupyter-Kernels. Damit ein Bild dargestellt wird, muss `show_image(...)` als **letzte Expression einer Zelle ohne Semikolon** stehen.

In [ ]:
#include "nlohmann/json.hpp"
#include "xtl/xbase64.hpp"

namespace nl = nlohmann;

namespace mvsr
{
    struct NotebookImage
    {
        std::string png_data;

        explicit NotebookImage(const cv::Mat& image)
        {
            std::vector<unsigned char> buffer;
            cv::imencode(".png", image, buffer);

            png_data.assign(
                reinterpret_cast<const char*>(buffer.data()),
                buffer.size()
            );
        }
    };

    nl::json mime_bundle_repr(const NotebookImage& image)
    {
        auto bundle = nl::json::object();
        bundle["image/png"] = xtl::base64encode(image.png_data);
        return bundle;
    }
}

mvsr::NotebookImage show_image(const cv::Mat& image)
{
    return mvsr::NotebookImage(image);
}

## 1. Was ist ein digitales Bild?

Ein digitales Bild ist zunächst nur ein Array bzw. eine Matrix aus Zahlen. Für ein Farbbild gilt typischerweise:

$
I \in \mathbb{R}^{H \times W \times C}
$

mit

- \(H\): Bildhöhe,
- \(W\): Bildbreite,
- \(C\): Anzahl der Farbkanäle.

In OpenCV wird ein Bild in C++ typischerweise durch `cv::Mat` repräsentiert. Die tatsächlich gespeicherten Werte hängen vom Datentyp ab. Bei `CV_8UC3` werden beispielsweise drei 8-Bit-Kanäle mit Werten von 0 bis 255 gespeichert.

Wir erzeugen zunächst eine einfache Testszene direkt mit OpenCV. Dadurch ist das Notebook unabhängig von externen Dateien.

Der folgende Code erzeugt eine einfache synthetische Testszene mit einer Auflösung von **640 × 480 Pixeln** und drei Farbkanälen. Alle Pixel erhalten zunächst den Wert 255, also Weiß.

- `cv::Mat(...)` – erzeugt bzw. verwaltet eine Bildmatrix.
- `cv::rectangle(...)` – zeichnet ein Rechteck in ein Bild.
- `cv::circle(...)` – zeichnet einen Kreis.
- `cv::Scalar(B,G,R)` – beschreibt hier eine Farbe in OpenCV-typischer **BGR-Reihenfolge**.
- `-1` als Linienstärke – füllt die jeweilige Form vollständig aus.
- `show_image(...)` – zeigt das Bild innerhalb des Browser-Notebooks an.

> **Binder-/Notebook-Hinweis:** Für die Darstellung wird hier `show_image(...)` verwendet. In einem lokalen C++-Programm würde das Bild normalerweise mit `cv::imshow()` und `cv::waitKey()` angezeigt.

In [ ]:
cv::Mat image(480, 640, CV_8UC3, cv::Scalar(255, 255, 255));

cv::rectangle(image, cv::Point(70, 100), cv::Point(240, 300),
              cv::Scalar(0, 0, 255), -1);       // Rot in BGR

cv::circle(image, cv::Point(380, 200), 90,
           cv::Scalar(0, 255, 0), -1);          // Grün

cv::rectangle(image, cv::Point(470, 300), cv::Point(600, 420),
              cv::Scalar(255, 0, 0), -1);       // Blau

show_image(image)

Der folgende Code gibt grundlegende Eigenschaften des Bildes wie Breite, Höhe, Anzahl der Kanäle und Datentyp aus und liest beispielhaft die Farbwerte einzelner Pixel aus.

In [ ]:
std::cout << "Breite: " << image.cols << std::endl;
std::cout << "Höhe: " << image.rows << std::endl;
std::cout << "Kanäle: " << image.channels() << std::endl;
std::cout << "Datentyp (OpenCV-Code): " << image.type() << std::endl;

cv::Vec3b pixel_00 = image.at<cv::Vec3b>(0, 0);
cv::Vec3b pixel_green = image.at<cv::Vec3b>(200, 380);

std::cout << "Pixel [0, 0] (B,G,R): "
          << static_cast<int>(pixel_00[0]) << ", "
          << static_cast<int>(pixel_00[1]) << ", "
          << static_cast<int>(pixel_00[2]) << std::endl;

std::cout << "Pixel [200, 380] (B,G,R): "
          << static_cast<int>(pixel_green[0]) << ", "
          << static_cast<int>(pixel_green[1]) << ", "
          << static_cast<int>(pixel_green[2]) << std::endl;

### Wichtig: Indizierung

OpenCV greift in C++ auf einen BGR-Pixel einer `CV_8UC3`-Matrix beispielsweise mit

```cpp
image.at<cv::Vec3b>(y, x)
```

zu.

- erster Index: **Zeile / y-Koordinate**
- zweiter Index: **Spalte / x-Koordinate**

Der Ursprung \((0,0)\) liegt oben links.

### Probieren Sie selbst

1. Lesen Sie einen Pixel aus dem roten Rechteck aus.
2. Lesen Sie einen Pixel aus dem blauen Rechteck aus.
3. Ändern Sie einen kleinen Bildbereich auf Weiß.
4. Zeigen Sie das veränderte Bild an.

> **Binder-/Notebook-Hinweis:** Die Schreibweise `[](){ ... }()` erzeugt eine kleine anonyme Funktion (Lambda), die direkt ausgeführt wird. Sie wird hier nur verwendet, damit die Variablen der Übungszelle lokal bleiben und die Zelle nach Änderungen leichter erneut ausgeführt werden kann. Gleichzeitig kann das Bild als letzte Expression an den Jupyter-Kernel zurückgegeben werden. In einem normalen lokalen C++-Programm wäre diese Lambda-Konstruktion dafür nicht nötig; dort würde man gewöhnlichen C++-Code bzw. eine normale Funktion verwenden und das Ergebnis mit `cv::imshow()` anzeigen.

In [ ]:
[]()
{
    cv::Mat image_modified = image.clone();

    // TODO: Pixel auslesen
    // cv::Vec3b pixel = image_modified.at<cv::Vec3b>(y, x);

    // TODO: kleinen Bildbereich auf Weiß setzen
    // cv::rectangle(image_modified, cv::Point(...), cv::Point(...),
    //               cv::Scalar(255, 255, 255), -1);

    return show_image(image_modified);
}()

## 2. Farbkanäle: BGR und RGB

Eine Kamera liefert pro Pixel mehrere Farbwerte. OpenCV verwendet für Farbbilder standardmäßig die Reihenfolge

$
[B,\;G,\;R].
$

Das ist wichtig, weil viele andere Bibliotheken und Bildformate Farben als RGB interpretieren.

RGB bzw. BGR basiert auf der **additiven Farbmischung**: Eine Farbe entsteht durch die Kombination unterschiedlicher Anteile von Rot, Grün und Blau. Sind alle drei Farbanteile maximal, entsteht Weiß; sind alle drei null, entsteht Schwarz.

Beispielsweise gilt für ein 8-Bit-Bild:

Schwarz: $[0, 0, 0]$

Weiß: $[255, 255, 255]$

Ein weißer Bildbereich besitzt daher in allen drei Farbkanälen hohe Werte. Wird das Bild anschließend in einzelne B-, G- und R-Kanäle zerlegt, erscheint der weiße Hintergrund entsprechend in allen drei Kanalbildern hell. Ein rein rotes Objekt erscheint dagegen hauptsächlich im Rot-Kanal hell.

Mit `cv::split()` können die drei Kanäle in einzelne Grauwertbilder zerlegt werden.

In [ ]:
std::vector<cv::Mat> bgr_channels;
cv::split(image, bgr_channels);

std::cout << "Blue Channel" << std::endl;
show_image(bgr_channels[0])

> **Binder-/Notebook-Hinweis:** In einem lokalen C++-Programm könnten die drei Kanäle mit drei `cv::imshow()`-Aufrufen gleichzeitig in eigenen Fenstern dargestellt werden. Die Rich-Display-Lösung in Binder zeigt dagegen das Ergebnis der letzten Expression einer Zelle. Deshalb werden die drei Kanäle hier in drei getrennten Zellen angezeigt.

In [ ]:
std::cout << "Blue Channel" << std::endl;
show_image(bgr_channels[0])

In [ ]:
std::cout << "Green Channel" << std::endl;
show_image(bgr_channels[1])

In [ ]:
std::cout << "Red Channel" << std::endl;
show_image(bgr_channels[2])

### Probieren Sie selbst

Setzen Sie einen Farbkanal vollständig auf Null und betrachten Sie das Resultat.

In C++ kann ein einzelner Kanal beispielsweise nach `cv::split()` mit

```cpp
channels[2].setTo(0);
```

auf Null gesetzt und anschließend mit `cv::merge()` wieder zu einem BGR-Bild zusammengesetzt werden.

Welche Farbe verschwindet? Welche Farben bleiben sichtbar?

> **Binder-/Notebook-Hinweis:** Auch hier wird eine unmittelbar ausgeführte Lambda-Funktion verwendet, damit lokale Variablen beim erneuten Ausführen der Übungszelle nicht mit bereits vorhandenen Variablen kollidieren. Lokal wäre diese Konstruktion nicht erforderlich.

In [ ]:
[]()
{
    cv::Mat test = image.clone();

    std::vector<cv::Mat> channels_test;
    cv::split(test, channels_test);

    // TODO: Wählen Sie einen Kanal und setzen Sie ihn auf 0.
    channels_test[2].setTo(0);

    cv::merge(channels_test, test);

    return show_image(test);
}()

## 3. HSV-Farbraum

RGB/BGR beschreibt eine Farbe über ihre Farbanteile. HSV trennt die Information anders:

- **H – Hue:** Farbton
- **S – Saturation:** Sättigung
- **V – Value:** Helligkeit

Das kann z.B. für einfache Farbsegmentierung nützlich sein.

Die Berechnung des HSV-Farbraums in OpenCV erfolgt auf Basis der RGB-Werte:

$V = \max(R,G,B)$

$S =
\begin{cases}
\dfrac{V-\min(R,G,B)}{V}, & \text{if } V \neq 0,\\
0, & \text{otherwise}
\end{cases}$

$H =
\begin{cases}
60\dfrac{G-B}{V-\min(R,G,B)}, & \text{if } V=R,\\
120 + 60\dfrac{B-R}{V-\min(R,G,B)}, & \text{if } V=G,\\
240 + 60\dfrac{R-G}{V-\min(R,G,B)}, & \text{if } V=B,\\
0, & \text{if } R=G=B
\end{cases}$

$H < 0 \;\Rightarrow\; H \leftarrow H + 360$

Für 8-Bit-Bilder werden die Werte anschließend skaliert:

$V \leftarrow 255V, \qquad
S \leftarrow 255S, \qquad
H \leftarrow \dfrac{H}{2}$

Damit liegt der Hue-Wert bei einem 8-Bit-HSV-Bild in OpenCV im Bereich \(0\dots179\).

Mit `cv::cvtColor()` wird das BGR-Bild in den HSV-Farbraum umgerechnet. Anschließend werden H, S und V mit `cv::split()` getrennt.

Der Hue-Wert ist bei sehr geringer Sättigung nicht sinnvoll interpretierbar. Das betrifft insbesondere weiße, graue und schwarze Bildbereiche. OpenCV speichert dort dennoch einen Hue-Wert. Für die Visualisierung werden deshalb Pixel mit \(S<10\) im Hue-Bild ausgeblendet.

Für die farbige Hue-Darstellung wird aus dem H-Kanal ein künstliches HSV-Bild mit voller Sättigung und Helligkeit aufgebaut und anschließend zurück nach BGR konvertiert.

In [ ]:
cv::Mat hsv;
cv::cvtColor(image, hsv, cv::COLOR_BGR2HSV);

std::vector<cv::Mat> hsv_channels;
cv::split(hsv, hsv_channels);

cv::Mat h = hsv_channels[0];
cv::Mat s = hsv_channels[1];
cv::Mat v = hsv_channels[2];

// Pixel mit sehr geringer Sättigung maskieren
cv::Mat low_saturation_mask;
cv::compare(s, 10, low_saturation_mask, cv::CMP_LT);

cv::Mat h_visual = h.clone();
h_visual.setTo(0, low_saturation_mask);

// Hue farbig darstellen: H beibehalten, S und V auf Maximum setzen
cv::Mat full_s(h.size(), CV_8UC1, cv::Scalar(255));
cv::Mat full_v(h.size(), CV_8UC1, cv::Scalar(255));

std::vector<cv::Mat> hue_display_channels = {h_visual, full_s, full_v};

cv::Mat hue_display_hsv;
cv::merge(hue_display_channels, hue_display_hsv);

cv::Mat hue_display_bgr;
cv::cvtColor(hue_display_hsv, hue_display_bgr, cv::COLOR_HSV2BGR);

// Maskierte Bereiche weiß darstellen
hue_display_bgr.setTo(cv::Scalar(255, 255, 255), low_saturation_mask);

> **Binder-/Notebook-Hinweis:** Wie bei den BGR-Kanälen werden H, S und V in Binder getrennt angezeigt, weil `show_image(...)` als letzte Expression einer Zelle stehen muss. Lokal könnten dafür mehrere `cv::imshow()`-Fenster verwendet werden.

### Hue

In [ ]:
show_image(hue_display_bgr)

### Saturation

In [ ]:
show_image(s)

### Value

In [ ]:
show_image(v)

Der HSV-Wert eines einzelnen Pixels kann wie beim BGR-Bild direkt mit `image.at<cv::Vec3b>(y,x)` ausgelesen werden.

In [ ]:
cv::Vec3b hsv_green = hsv.at<cv::Vec3b>(200, 380);

std::cout << "HSV-Wert im grünen Kreis: "
          << static_cast<int>(hsv_green[0]) << ", "
          << static_cast<int>(hsv_green[1]) << ", "
          << static_cast<int>(hsv_green[2]) << std::endl;

### Einfache Farbsegmentierung

Wir wollen nun nur das **grüne Objekt** auswählen. Dazu definieren wir einen Bereich im HSV-Raum und erzeugen mit `cv::inRange()` eine Binärmaske.

In [ ]:
// Definieren der Limits im HSV-Farbraum
cv::Scalar lower_green(45, 100, 100);
cv::Scalar upper_green(85, 255, 255);

// Maske für Pixelwerte innerhalb des definierten Bereichs
cv::Mat mask_green;
cv::inRange(hsv, lower_green, upper_green, mask_green);

// Segmentiertes Objekt
cv::Mat segmented_green;
cv::bitwise_and(image, image, segmented_green, mask_green);

> **Binder-/Notebook-Hinweis:** Maske und segmentiertes Bild werden im Notebook in zwei Zellen dargestellt. In einem lokalen C++-Programm könnten beide mit zwei `cv::imshow()`-Aufrufen gleichzeitig angezeigt werden.

### Maske für grüne Bildbereiche

In [ ]:
show_image(mask_green)

### Segmentiertes grünes Objekt

In [ ]:
show_image(segmented_green)

### Mini-Aufgabe: Segmentieren Sie eine andere Farbe

Passen Sie die HSV-Grenzen so an, dass statt des grünen Kreises

- das blaue Rechteck **oder**
- das rote Rechteck

segmentiert wird.

> **Binder-/Notebook-Hinweis:** Die Berechnung wird in einer Lambda-Funktion ausgeführt, damit die Grenzwerte beim Experimentieren mehrfach verändert und die Zelle erneut ausgeführt werden kann. Die beiden Ergebnisbilder werden danach getrennt angezeigt. In einem lokalen C++-Programm wäre dafür keine Lambda-Konstruktion nötig.

In [ ]:
cv::Mat mask_custom;
cv::Mat result_custom;

In [ ]:
[]()
{
    // TODO: HSV-Grenzen festlegen
    cv::Scalar lower(0, 0, 0);
    cv::Scalar upper(0, 255, 255);

    cv::inRange(hsv, lower, upper, mask_custom);
    cv::bitwise_and(image, image, result_custom, mask_custom);
}()

### Ihre Maske

In [ ]:
show_image(mask_custom)

### Ihre Segmentierung

In [ ]:
show_image(result_custom)

## 4. Intrinsische Kameraparameter

Für die Projektion eines 3D-Punktes in das Bild benötigen wir die intrinsische Kameramatrix

$
K =
\begin{bmatrix}
f_x & 0 & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}.
$

Dabei beschreiben

- \(f_x, f_y\): Brennweite in Pixel,
- \(c_x, c_y\): Hauptpunkt des Bildes.

Für dieses Beispiel nehmen wir an, dass die Kamera bereits kalibriert wurde.

In [ ]:
double fx = 600.0;
double fy = 600.0;
double cx = 320.0;
double cy = 240.0;

cv::Matx33d K(
    fx, 0.0, cx,
    0.0, fy, cy,
    0.0, 0.0, 1.0
);

std::cout << "K =\n" << cv::Mat(K) << std::endl;

### 3D → 2D Projektion

Ein Punkt im Kamerakoordinatensystem

$
P_c = \begin{bmatrix} X \\ Y \\ Z \end{bmatrix}
$

wird auf einen Pixel \((u,v)\) projiziert. Für das einfache Lochkameramodell gilt:

$
u = f_x \frac{X}{Z} + c_x
$

$
v = f_y \frac{Y}{Z} + c_y.
$

In [ ]:
double X = 0.10;
double Y = 0.05;
double Z = 1.00;

double u = fx * X / Z + cx;
double v_img = fy * Y / Z + cy;

std::cout << std::fixed << std::setprecision(3);
std::cout << "3D-Punkt: X=" << X
          << " m, Y=" << Y
          << " m, Z=" << Z << " m" << std::endl;

std::cout << std::setprecision(2);
std::cout << "Projizierter Pixel: u=" << u
          << ", v=" << v_img << std::endl;

### Probieren Sie selbst

Verändern Sie einzeln \(X\), \(Y\), \(Z\), \(f_x\) oder \(f_y\) und beobachten Sie, wie sich der Bildpunkt verändert.

Überlegen Sie **vor** dem Ausführen:

- Was passiert mit \(u\), wenn \(X\) größer wird?
- Was passiert mit dem Abstand zum Hauptpunkt, wenn \(Z\) größer wird?
- Welche Wirkung hat eine größere Brennweite?

> **Binder-/Notebook-Hinweis:** Die geschweiften Klammern erzeugen hier nur einen lokalen Gültigkeitsbereich. Dadurch können die Testvariablen beim erneuten Ausführen der Zelle wieder angelegt werden. In einem normalen C++-Programm wäre dieser zusätzliche Block für das Beispiel nicht notwendig.

In [ ]:
{
    double X_test = 0.10;
    double Y_test = 0.05;
    double Z_test = 1.00;
    double fx_test = fx;
    double fy_test = fy;

    double u_test = fx_test * X_test / Z_test + cx;
    double v_test = fy_test * Y_test / Z_test + cy;

    std::cout << std::fixed << std::setprecision(2)
              << "u=" << u_test
              << ", v=" << v_test << std::endl;
}

## 5. Tiefe als zusätzliche Information

Ein normales Kamerabild liefert zunächst 2D-Bildkoordinaten \((u,v)\). Ein Tiefensensor ergänzt für einen Bildpunkt einen Tiefenwert \(Z\):

$
(u,v) \rightarrow (u,v,Z)
$

Mit der Kameramatrix kann daraus ein 3D-Punkt im Kamerakoordinatensystem rekonstruiert werden.

Wir erzeugen dafür eine synthetische Tiefenkarte. Der Hintergrund liegt bei 1,5 m, ein kreisförmiger Bereich bei 0,8 m.

In [ ]:
cv::Mat depth(480, 640, CV_32FC1, cv::Scalar(1.5f));

cv::Mat depth_mask = cv::Mat::zeros(depth.size(), CV_8UC1);
cv::circle(depth_mask, cv::Point(380, 200), 90, cv::Scalar(255), -1);
depth.setTo(0.8f, depth_mask);

// Nur für die Visualisierung auf 0...255 skalieren
cv::Mat depth_vis;
cv::normalize(depth, depth_vis, 0, 255, cv::NORM_MINMAX, CV_8UC1);

// Pseudofarbdarstellung ähnlich zur viridis-Darstellung in Python
cv::Mat depth_color;
cv::applyColorMap(depth_vis, depth_color, cv::COLORMAP_VIRIDIS);

Die Matrix `depth` enthält weiterhin die tatsächlichen Tiefenwerte in Metern. `depth_vis` und `depth_color` dienen ausschließlich der Darstellung.

> **Binder-/Notebook-Hinweis:** Die Tiefenkarte wird mit `show_image(...)` direkt im Notebook dargestellt. Lokal würde man für die Pseudofarbdarstellung typischerweise `cv::imshow("Synthetische Tiefenkarte", depth_color)` verwenden. Da die Notebook-Darstellung als letzte Expression einer Zelle erfolgen muss, sind Visualisierung und Textausgabe hier getrennt.

In [ ]:
show_image(depth_color)

In [ ]:
std::cout << std::fixed << std::setprecision(2);
std::cout << "Tiefe im Hintergrund: "
          << depth.at<float>(50, 50) << " m" << std::endl;

std::cout << "Tiefe im Kreis: "
          << depth.at<float>(200, 380) << " m" << std::endl;

### 2D + Tiefe → 3D

Aus Pixelkoordinaten und Tiefe folgt für das Lochkameramodell:

$
X = \frac{(u-c_x)Z}{f_x}
$

$
Y = \frac{(v-c_y)Z}{f_y}.
$

Damit wird aus einem Pixel wieder ein 3D-Punkt relativ zur Kamera.

In [ ]:
double u_depth = 380.0;
double v_depth = 200.0;

double Z_depth = static_cast<double>(
    depth.at<float>(
        static_cast<int>(v_depth),
        static_cast<int>(u_depth)
    )
);

double X_depth = (u_depth - cx) * Z_depth / fx;
double Y_depth = (v_depth - cy) * Z_depth / fy;

cv::Vec3d point_3d(X_depth, Y_depth, Z_depth);

std::cout << "Pixel (u, v): ("
          << u_depth << ", " << v_depth << ")" << std::endl;

std::cout << std::fixed << std::setprecision(2)
          << "Tiefe Z: " << Z_depth << " m" << std::endl;

std::cout << std::setprecision(3)
          << "3D-Punkt [X, Y, Z]: ["
          << point_3d[0] << ", "
          << point_3d[1] << ", "
          << point_3d[2] << "] m" << std::endl;

## 6. Mini-Aufgabe: Hin und zurück

Wählen Sie selbst einen gültigen Pixel \((u,v)\) aus der Tiefenkarte.

1. Lesen Sie dessen Tiefenwert \(Z\) aus.
2. Rekonstruieren Sie daraus \((X,Y,Z)\).
3. Projizieren Sie diesen 3D-Punkt wieder in das Bild.
4. Vergleichen Sie den resultierenden Pixel mit dem ursprünglichen Pixel.

Wenn das Kameramodell konsistent angewendet wurde, sollten beide Bildpunkte nahezu identisch sein.

> **Binder-/Notebook-Hinweis:** Der zusätzliche Block `{ ... }` hält die Variablen dieser Übung lokal, damit die Zelle beim Experimentieren einfacher erneut ausgeführt werden kann. In einem normalen lokalen C++-Programm wäre dieser Block für die Rechnung nicht notwendig.

In [ ]:
{
    // TODO: Eigenen Pixel wählen
    double u0 = 300.0;
    double v0 = 250.0;

    double Z0 = static_cast<double>(
        depth.at<float>(
            static_cast<int>(v0),
            static_cast<int>(u0)
        )
    );

    double X0 = (u0 - cx) * Z0 / fx;
    double Y0 = (v0 - cy) * Z0 / fy;

    double u_back = fx * X0 / Z0 + cx;
    double v_back = fy * Y0 / Z0 + cy;

    double error_px = std::sqrt(
        (u_back - u0) * (u_back - u0) +
        (v_back - v0) * (v_back - v0)
    );

    std::cout << std::fixed << std::setprecision(3);
    std::cout << "Original: (" << u0 << ", " << v0 << ")" << std::endl;
    std::cout << "3D-Punkt: (" << X0 << ", " << Y0 << ", " << Z0 << ")" << std::endl;
    std::cout << "Rückprojektion: (" << u_back << ", " << v_back << ")" << std::endl;
    std::cout << "Fehler [Pixel]: " << error_px << std::endl;
}

## 7. Selbstcheck

Beantworten Sie die Fragen zunächst ohne in die Folien zu schauen.

1. Warum wird beim Pixelzugriff zuerst \(y\) und danach \(x\) angegeben?
2. In welcher Kanalreihenfolge speichert OpenCV ein Farbbild standardmäßig?
3. Warum kann HSV für Farbsegmentierung praktisch sein?
4. Welche Bedeutung haben \(f_x, f_y, c_x, c_y\)?
5. Warum reicht ein einzelner Pixel \((u,v)\) nicht aus, um einen eindeutigen 3D-Punkt zu bestimmen?
6. Was passiert mit der Bildposition eines Punktes, wenn seine Tiefe \(Z\) größer wird, \(X\) und \(Y\) aber gleich bleiben?

<details>
<summary><b>Kurze Antworten anzeigen</b></summary>

1. Ein Bild ist als Zeilen und Spalten organisiert. Daher wird zuerst die Zeile \(y\), danach die Spalte \(x\) angegeben.
2. BGR.
3. Farbton, Sättigung und Helligkeit sind getrennt. Dadurch können bestimmte Farben oft einfacher über Schwellwerte ausgewählt werden.
4. \(f_x,f_y\) beschreiben die Brennweite in Pixel; \(c_x,c_y\) den Hauptpunkt.
5. Entlang des Sehstrahls liegen unendlich viele mögliche 3D-Punkte. Es fehlt die Tiefe.
6. Der Punkt wandert näher zum Hauptpunkt, da \(X/Z\) und \(Y/Z\) kleiner werden.

</details>

## 8. Take-away

Für die Präsenz-LV sollten Sie folgende Punkte mitnehmen:

- Ein Sensor liefert zunächst **numerische Messwerte**, keine Objektinformation.
- Ein digitales Bild ist eine Matrix aus Pixelwerten; OpenCV verwendet dafür in C++ typischerweise `cv::Mat`.
- Die Wahl des Farbraums beeinflusst, wie leicht bestimmte Bildinformationen verarbeitet werden können.
- Die intrinsische Kameramatrix beschreibt die Abbildung von Kamerakoordinaten auf Pixelkoordinaten.
- Ein Tiefenwert ergänzt die 2D-Bildinformation um die Information, die für eine Rekonstruktion eines 3D-Punktes benötigt wird.

### Ausblick

In der ersten Hausübung führen Sie anschließend selbstständig eine **Kamerakalibrierung mit OpenCV 4.12** durch und bestimmen die intrinsischen Kameraparameter Ihrer Kamera.